# W3: SEO 키워드 분석

상품명/카테고리를 입력해 경쟁 키워드 10개와 경쟁도 점수를 생성합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

product_name = "프리미엄 스마트워치"
category = "전자기기"


def analyze(prompt):
    fallback = "".join([
        "무선이어폰,전자기기,베스트셀러,할인행사,상품평
",
    ])
    return use_gemini(prompt, fallback)

prompt = (
    f"상품명:{product_name}, 카테고리:{category} 기준으로 경쟁 키워드 10개를 [키워드, 경쟁도 점수1~10] 형태로 만들어줘."
)
text = analyze(prompt)
print(text)


In [ ]:
# 결과 보조 표기
keywords = pd.DataFrame({
    "키워드": ["스마트워치", "블루투스워치", "아이웨어", "운동용시계", "남성시계", "여성시계", "방수시계", "배터리", "할인", "기획전"],
    "경쟁도": [8, 7, 6, 7, 6, 5, 4, 8, 9, 6]
})
print(keywords)
